# **Day 73: More asyncio - Timeouts, Limits and Real Web Requests**

In [1]:
import asyncio, time, random

## **Timeouts: Do Not Wait Forever**
If a server is very slow, we should stop waiting after some time.

In [2]:
async def slow_server():
    await asyncio.sleep(3)
    return "finally"

try:
    result = await asyncio.wait_for(slow_server(), timeout=1)
except TimeoutError:
    print("The server took too long, we stopped waiting after 1 second")

The server took too long, we stopped waiting after 1 second


## **Errors in gather**
With `return_exceptions=True`, an error in one job does not stop the others. The error is returned as a value in the list instead.

In [3]:
async def check_station(code):
    await asyncio.sleep(0.1)
    if code == "XYZ":
        raise ValueError(f"Unknown station {code}")
    return f"{code} OK"

results = await asyncio.gather(*(check_station(c) for c in ["HWH", "XYZ", "NJP"]), return_exceptions=True)
for r in results:
    print(type(r).__name__, "->", r)

str -> HWH OK
ValueError -> Unknown station XYZ
str -> NJP OK


## **Semaphore: Limit How Many Run at Once**
When downloading from a website, we must **not** send 100 requests at the same moment. That is rude and the website may block us. A **Semaphore** allows only a fixed number of jobs at a time.

**Real-life example:** A shop that allows only **3 customers inside** at a time. Others wait at the door.

In [4]:
limit = asyncio.Semaphore(3)
active = 0
peak = 0

async def polite_download(i):
    global active, peak
    async with limit:                    # Only 3 can be inside this block at once
        active += 1
        peak = max(peak, active)
        await asyncio.sleep(random.uniform(0.1, 0.3))
        active -= 1
        return i

random.seed(1)
start = time.perf_counter()
done = await asyncio.gather(*(polite_download(i) for i in range(12)))
print(f"{len(done)} tiles downloaded, never more than {peak} at a time, in {time.perf_counter() - start:.2f} s")

12 tiles downloaded, never more than 3 at a time, in 0.89 s


## **TaskGroup (Python 3.11+)**
A modern and safe way to run many tasks. If one task fails, the others are cancelled automatically.

In [5]:
async def job(name, seconds):
    await asyncio.sleep(seconds)
    return f"{name} done"

async with asyncio.TaskGroup() as tg:
    t1 = tg.create_task(job("download", 0.3))
    t2 = tg.create_task(job("resize", 0.2))

print(t1.result(), "|", t2.result())

download done | resize done


## **Example from GIS: Getting Real Elevations from the Internet**
**Elevation** is the height of a place above sea level. It comes from a **DEM (Digital Elevation Model)**, which is a satellite-based map of heights.

The free **Open-Meteo** website has an elevation service. Let us ask it for the height of several places in West Bengal **at the same time**.

We use the `requests` library inside `asyncio.to_thread()`, which lets a normal (blocking) function run in the background without freezing the event loop.

In [6]:
import requests

places = {
    "Kolkata": (22.5726, 88.3639),
    "Digha": (21.6266, 87.5074),
    "Midnapore": (22.4257, 87.3199),
    "Purulia": (23.3321, 86.3652),
    "Siliguri": (26.7271, 88.3953),
    "Kurseong": (26.8800, 88.2780),
    "Darjeeling": (27.0410, 88.2663),
    "Sandakphu": (27.1060, 88.0020),
}

def get_elevation(name, lat, lon):
    r = requests.get("https://api.open-meteo.com/v1/elevation",
                     params={"latitude": lat, "longitude": lon}, timeout=15)
    r.raise_for_status()
    return name, r.json()["elevation"][0]

start = time.perf_counter()
try:
    results = await asyncio.gather(*(asyncio.to_thread(get_elevation, n, *ll) for n, ll in places.items()))
    for name, height in sorted(results, key=lambda x: x[1]):
        bar = "#" * int(height // 100)
        print(f"{name:<11}{height:>7.0f} m  {bar}")
except Exception as e:
    print("Network problem:", e)
print(f"\n{len(places)} requests in {time.perf_counter() - start:.2f} seconds")

Digha           10 m  
Kolkata         12 m  
Midnapore       48 m  
Siliguri       126 m  #
Purulia        253 m  ##
Kurseong      1498 m  ##############
Darjeeling    2100 m  #####################
Sandakphu     3567 m  ###################################

8 requests in 0.74 seconds


These are **real values** from a DEM, fetched live when this notebook was run. We can clearly see how the land rises from the coast (Digha) to the plains (Kolkata), the Chota Nagpur plateau edge (Purulia), and then the Himalayas (Darjeeling and Sandakphu, the highest point of West Bengal).

(The DEM gives the height of a grid cell around the point, so the numbers can be a little different from the heights written on signboards.)

In [7]:
# Quick question
# Why should we limit the number of requests sent to a website at the same time?